# Novo Nordisk Grant Applicant to OpenAlex Author Matching

This notebook is a coding sample extracted from a research data project at the Knowledge Lab at the University of Chicago, where I work as one of the primary data analysts on this project. The broader empirical goal of the project is to study whether funded research proposals ultimately lead to the realization of the research agendas described in those proposals.

To answer that question, we need to connect grant applications to the later publication records of the applicants. This notebook focuses on one core part of that pipeline: matching Novo Nordisk grant applicants to OpenAlex author IDs.

The unit of observation is a grant application. The desired output is either one OpenAlex author ID for the main applicant or no match when the evidence is not strong enough. The notebook does not try to maximize the number of matches at all costs; it is written to avoid assigning an applicant to the wrong author.

The matching logic uses two evidence sources, ordered from highest to lowest confidence:

1. ORCID identifiers from applicant records.
2. Literature references from applications matched to OpenAlex works.

ORCID matches are the strongest signal because ORCID is designed as a unique author identifier. Literature-reference matches are useful when an applicant does not have an accessible ORCID. In that case, I match a cited publication title to an OpenAlex work, inspect the authorship list for that work, and only accept the match if the applicant can be identified conservatively among the authors.

Applicant names are used only for disambiguation after a cited work has already been matched. 

The matching logic can be summarized as:

```text
ORCID path
  application ORCID
    -> normalize ORCID
    -> join to OpenAlex author ORCID
    -> accept as match_type = "orcid"

Reference path
  extracted reference title from applications dataset
    -> normalize title
    -> join to normalized OpenAlex works title
    -> grab that work's authorship list
    -> require applicant last name in author list
    -> require first-initial agreement
    -> if still ambiguous, require full first-name agreement
    -> accept only if exactly one author remains
    -> otherwise reject

Final decision
  ORCID matches and reference matches
    -> apply confidence ordering: orcid > refs_publication
    -> keep one author match per application
    -> leave weak or ambiguous cases unmatched
```

In [1]:
# Import libraries and set parameters
from __future__ import annotations

import re
import unicodedata
from collections.abc import Mapping, Sequence
from typing import TypedDict

import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(100)

ORCID_PATTERN = r"(?i)(\d{4}-\d{4}-\d{4}-\d{3}[0-9X])"
ORCID_RE = re.compile(ORCID_PATTERN)

MATCH_PRIORITY = {
    "orcid": 1,
    "refs_publication": 2,
}

## Data description

The full project combines two primary data sources. I include only the relevant columns in the tables below.

**Novo Nordisk application records** contain applicant identifiers, names, application metadata, proposal text, CV information, ORCID identifiers when available, and reference lists included in the application. These records describe what the applicant proposed at the time of application. Because these data are confidential, I use synthetic data throughout to illustrate the matching process. In the full project, columns such as `Main applicant first name`, `Main applicant last name`, `Project literature references`, and `Literature references` are renamed into simplified analysis columns such as first_name, last_name, and reference_titles.

**OpenAlex records** provide the bibliographic side of the project. The authors table contains author-level identifiers and metadata, while the works table contains publication-level records, including titles and nested authorship lists. Those authorship lists are what allow the literature-reference match to move from a cited work to a candidate OpenAlex author ID.

The tables below describe the fields used in the sample. The application table preserves the schema of the source `applications` parquet, but all example values are synthetic. The OpenAlex author and work examples are drawn from small sample parquet files.

### Anonymized `applications` sample schema

| Field | Type | Example value |
| --- | --- | --- |
| `Application reference numeric` | `Int64` | 1001 |
| `Main applicant id` | `String` | APP-001 |
| `Main applicant first name` | `String` | Beth |
| `Main applicant last name` | `String` | Lopez-Marques |
| `Project title` | `String` | Network formation in biomedical research |
| `Project literature references` | `String` | Lopez-Marques, B. and Chen, R. Networks and Science. Journal of Research. |
| `Literature references` | `String` | Lopez-Marques, B. and Chen, R. Networks and Science. Journal of Research. |
| `Applicant CV` | `String` | Example CV text. ORCID: 0000-0002-1825-0097. |
| `application_year` | `Int32` | 2021 |

### Sample OpenAlex `authors` row

| Field | Type | Example value |
| --- | --- | --- |
| `oa_author_id` | `String` | https://openalex.org/A5079928590 |
| `display_name` | `String` | Annisa S. Riska |
| `display_name_alternatives` | `List(String)` | ['Annisa S. Riska'] |
| `name_norm` | `String` | annisa s riska |
| `orcid_norm` | `String` | null |

### Sample OpenAlex `works` row

| Field | Type | Example value |
| --- | --- | --- |
| `id` | `String` | https://openalex.org/W1489925047 |
| `title` | `String` | Remote power and console management in large datacenters |
| `publication_year` | `Int64` | 2010 |
| `authorships` | `List(Struct({'author': Struct({'display_name': String, 'id': String, 'orcid': String}), 'author_position': String, 'countries': List(String), 'institutions': List(Struct({'country_code': String, 'display_name': String, 'id': String, 'lineage': List(String), 'ror': String, 'type': String})), 'is_corresponding': Boolean, 'raw_affiliation_string': String, 'raw_affiliation_strings': List(String), 'raw_author_name': String}))` | [{'author': {'display_name': 'A. Horvath', 'id': 'https://openalex.org/A5016033896', 'orcid': None}, 'author_position': 'first', 'countries... |
| `title_norm` | `String` | remote power and console management in large datacenters |

## 1. Synthetic example data

The example data is synthetic and small on purpose. Each application row is designed to show one behavior of the matching logic.

| Application | Evidence available | Expected outcome |
| --- | --- | --- |
| `1` | Applicant has an ORCID that appears in the author table. | Match to `A1` through `orcid`. |
| `2` | Applicant has no ORCID, but cites a work where the author list identifies them clearly. | Match to `A3` through `refs_publication`. |
| `3` | Applicant cites a work with two indistinguishable `A. Smith` authors. | No match; the case is ambiguous. |
| `4` | Applicant cites a title that does not appear in the works table. | No match. |
| `5` | Applicant has no ORCID and no parsed reference titles. | No match. |
| `6` | Applicant cites a matched work, but the author with the same last name has the wrong first initial. | No match. |
| `7` | Applicant cites multiple titles; one title does not match, but another clearly identifies the applicant. | Match to `A6` through `refs_publication`. |

The reference rows below are derived from the application records. In the full project, this is the same idea: references begin inside the application text and are then parsed into one row per cited title for matching. I define this data before the functions so the reader can see the shape of each table first.

In [2]:
sample_cases = pl.DataFrame(
    [
        {
            "application_id": 1,
            "evidence": "ORCID in application and author table",
            "expected": "match A1 through orcid",
        },
        {
            "application_id": 2,
            "evidence": "reference title maps to a work with one clear applicant author",
            "expected": "match A3 through refs_publication",
        },
        {
            "application_id": 3,
            "evidence": "reference title maps to a work with ambiguous initials",
            "expected": "no match",
        },
        {
            "application_id": 4,
            "evidence": "reference title is not found in OpenAlex works",
            "expected": "no match",
        },
        {
            "application_id": 5,
            "evidence": "no ORCID and no parsed reference titles",
            "expected": "no match",
        },
        {
            "application_id": 6,
            "evidence": "matched work has a last-name hit but first initial does not match",
            "expected": "no match",
        },
        {
            "application_id": 7,
            "evidence": "one bad reference title and one clear reference title",
            "expected": "match A6 through refs_publication",
        },
    ]
)

# applications data (synthetic to protect privacy)
applications = pl.DataFrame(
    [
        {
            "application_id": 1,
            "first_name": "Alice",
            "last_name": "Smith",
            "orcid": "https://orcid.org/0000-0002-1825-0097",
            "reference_titles": [],
        },
        {
            "application_id": 2,
            "first_name": "Beth",
            "last_name": "Lopez-Marques",
            "orcid": None,
            "reference_titles": ["Networks and Science"],
        },
        {
            "application_id": 3,
            "first_name": "Anne",
            "last_name": "Smith",
            "orcid": None,
            "reference_titles": ["Ambiguous Methods"],
        },
        {
            "application_id": 4,
            "first_name": "Carl",
            "last_name": "Jones",
            "orcid": None,
            "reference_titles": ["Missing Work Title"],
        },
        {
            "application_id": 5,
            "first_name": "Dana",
            "last_name": "Miller",
            "orcid": None,
            "reference_titles": [],
        },
        {
            "application_id": 6,
            "first_name": "Nora",
            "last_name": "Patel",
            "orcid": None,
            "reference_titles": ["Initial Mismatch Study"],
        },
        {
            "application_id": 7,
            "first_name": "Eve",
            "last_name": "Chen",
            "orcid": None,
            "reference_titles": ["Unmatched Reference", "Cell Signals"],
        },
    ]
)

# authors table
authors = pl.DataFrame(
    [
        {"oa_author_id": "A1", "display_name": "Alice Smith", "orcid": "0000-0002-1825-0097"},
        {"oa_author_id": "A2", "display_name": "Alex Smith", "orcid": None},
        {"oa_author_id": "A3", "display_name": "Beth Lopez Marques", "orcid": None},
        {"oa_author_id": "A4", "display_name": "Peter Patel", "orcid": None},
        {"oa_author_id": "A5", "display_name": "Riley Morgan", "orcid": None},
        {"oa_author_id": "A6", "display_name": "Eve Chen", "orcid": None},
    ]
)

# Explode application reference lists so that there is one row per title
references = (
    applications.select(["application_id", "reference_titles"])
    .explode("reference_titles")
    .rename({"reference_titles": "title"})
    .filter(pl.col("title").is_not_null())
)

# works table
works = pl.DataFrame(
    [
        {
            "title": "Networks and Science",
            "authorships": [
                {
                    "raw_author_name": "B. Lopez Marques",
                    "author": {"id": "A3", "display_name": "Beth Lopez Marques"},
                }
            ],
        },
        {
            "title": "Ambiguous Methods",
            "authorships": [
                {"raw_author_name": "A. Smith", "author": {"id": "A1", "display_name": "A. Smith"}},
                {"raw_author_name": "A. Smith", "author": {"id": "A2", "display_name": "A. Smith"}},
            ],
        },
        {
            "title": "Initial Mismatch Study",
            "authorships": [
                {"raw_author_name": "Peter Patel", "author": {"id": "A4", "display_name": "Peter Patel"}},
            ],
        },
        {
            "title": "Cell Signals",
            "authorships": [
                {"raw_author_name": "E. Chen", "author": {"id": "A6", "display_name": "Eve Chen"}},
            ],
        },
    ]
)

sample_cases

application_id,evidence,expected
i64,str,str
1,"""ORCID in application and autho…","""match A1 through orcid"""
2,"""reference title maps to a work…","""match A3 through refs_publicat…"
3,"""reference title maps to a work…","""no match"""
4,"""reference title is not found i…","""no match"""
5,"""no ORCID and no parsed referen…","""no match"""
6,"""matched work has a last-name h…","""no match"""
7,"""one bad reference title and on…","""match A6 through refs_publicat…"


## 2. Normalization helpers

In [3]:
def normalize_name(value: object) -> str:
    """
    Normalize names: make lowercase, remove accents/punctuation, strip whitespace.
    """
    if value is None:
        return ""

    text = unicodedata.normalize("NFKD", str(value).lower())
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)  # hyphens/quotes/etc -> spaces
    return re.sub(r"\s+", " ", text).strip()


def normalize_orcid(value: object) -> str | None:
    """
    Extract and normalize an ORCID from arbitrary text.
    """
    if value is None:
        return None
    match = ORCID_RE.search(str(value))
    return match.group(1).upper() if match else None


def normalize_title(value: object) -> str | None:
    """
    Normalize publication titles for matching.
    """
    if not isinstance(value, str):
        return None

    text = value.lower()

    # remove trailing year
    text = re.sub(r"[\(\[]?\b(19|20)\d{2}\b[\)\]]?\s*$", "", text)

    # normalize separators (hyphens, colons, slashes) to spaces
    text = re.sub(r"[-:–—/]", " ", text)

    # remove remaining punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text or None

## 3. Author disambiguation

In [4]:
class AuthorCandidate(TypedDict):
    author_id: str
    author_name: str


def author_candidates_from_authorships(authorships: object) -> list[AuthorCandidate]:
    """
    Extract author IDs and display names from an OpenAlex authorship list.

    This is used in the references matching process, as titles from the references are 
    matched to the papers in works. The works table has authors listed in the nested
    "authorships" column. 
    """
    if not isinstance(authorships, list):
        return []

    candidates: list[AuthorCandidate] = []
    for authorship in authorships:
        if not isinstance(authorship, Mapping):
            continue

        author = authorship.get("author") or {}
        if not isinstance(author, Mapping):
            author = {}

        author_id = author.get("id") or authorship.get("author_id")
        author_name = (
            authorship.get("raw_author_name")
            or author.get("display_name")
            or authorship.get("author_name")
        )
        if author_id and author_name:
            candidates.append({"author_id": str(author_id), "author_name": str(author_name)})

    return candidates


def _author_first_name(author_name: str, last_tokens: set[str]) -> str:
    """
    Extract the first name portion of an author name.
    """
    tokens = normalize_name(author_name).split()
    return " ".join(token for token in tokens if token not in last_tokens)


def disambiguate_author(
    applicant_first: object,
    applicant_last: object,
    candidates: Sequence[AuthorCandidate],
) -> list[str]:
    """
    Match an applicant name to author IDs from one paper.

    The function returns a single author ID only when the match is confident.
    It first filters by last name, then by first initial, and finally by full
    first name if multiple candidates remain. Ambiguous or missing matches
    return an empty list.
    """
    first_norm = normalize_name(applicant_first)
    last_tokens = set(normalize_name(applicant_last).split())
    if not last_tokens or not candidates:
        return []

    # Last name must match
    last_name_matches = [
        candidate
        for candidate in candidates
        if last_tokens & set(normalize_name(candidate["author_name"]).split())
    ]
    if not last_name_matches:
        return []

    # Check for first initial match
    initial_matches = []
    for candidate in last_name_matches:
        author_first = _author_first_name(candidate["author_name"], last_tokens)
        if not first_norm or not author_first or author_first[0] == first_norm[0]:
            initial_matches.append(candidate)

    if len(initial_matches) == 1:
        return [initial_matches[0]["author_id"]]
    if not initial_matches:
        return []

    # If multiple last name + first initial matches, check with full first name
    full_name_matches = []
    for candidate in initial_matches:
        author_first = _author_first_name(candidate["author_name"], last_tokens)
        if not first_norm or not author_first or author_first == first_norm:
            full_name_matches.append(candidate)

    if len(full_name_matches) == 1:
        return [full_name_matches[0]["author_id"]]

    # Still ambiguous after all filters — no confident match
    return []

## 4. Match source functions

The next cell builds the two match sources. Each function returns the same basic schema: `application_id`, `oa_author_id`, and `match_type`. Keeping the outputs aligned makes the final priority step simple and auditable.

In [5]:
def match_orcid(
    applications: pl.DataFrame,
    authors: pl.DataFrame,
) -> pl.DataFrame:
    """
    Match applications to authors using normalized ORCID values.
    """
    # Normalize ORCIDs on both sides before merging
    apps = (
        applications.select(["application_id", "orcid"])
        .with_columns(
            pl.col("orcid").map_elements(normalize_orcid, return_dtype=pl.Utf8).alias("_orcid")
        )
        .filter(pl.col("_orcid").is_not_null())
    )

    # Do the same normalization for the author table
    author_ids = (
        authors.select(["oa_author_id", "orcid"])
        .with_columns(
            pl.col("orcid").map_elements(normalize_orcid, return_dtype=pl.Utf8).alias("_orcid")
        )
        .filter(pl.col("_orcid").is_not_null())
    )

    # Shared ORCID is enough to qualify as a match
    return (
        apps.join(author_ids, on="_orcid", how="inner")
        .select(
            "application_id",
            "oa_author_id",
            pl.lit("orcid").alias("match_type"),
        )
        .unique()
    )


def match_reference_works(
    references: pl.DataFrame,
    works: pl.DataFrame,
    applications: pl.DataFrame,
) -> pl.DataFrame:
    """
    Match extracted reference titles to works, then disambiguate authors.
    """
    # Reference titles and OpenAlex work titles are normalized into the same key
    refs = (
        references.select(["application_id", "title"])
        .with_columns(
            pl.col("title").map_elements(normalize_title, return_dtype=pl.Utf8).alias("_title_norm")
        )
        .filter(pl.col("_title_norm").is_not_null())
    )

    works_norm = (
        works.select(["title", "authorships"])
        .with_columns(
            pl.col("title").map_elements(normalize_title, return_dtype=pl.Utf8).alias("_title_norm")
        )
        .filter(pl.col("_title_norm").is_not_null())
    )

    # Applicant names are only used after a reference title has matched a work
    applicants = applications.select(
        "application_id",
        pl.col("first_name").map_elements(normalize_name, return_dtype=pl.Utf8).alias("_first_norm"),
        pl.col("last_name").map_elements(normalize_name, return_dtype=pl.Utf8).alias("_last_norm"),
    )

    joined = (
        refs.join(works_norm, on="_title_norm", how="inner")
        .join(applicants, on="application_id", how="left")
        .with_columns(
            # Disambiguate names within the matched work's list of authors
            pl.struct(["_first_norm", "_last_norm", "authorships"])
            .map_elements(
                lambda row: disambiguate_author(
                    row["_first_norm"],
                    row["_last_norm"],
                    author_candidates_from_authorships(row["authorships"]),
                ),
                return_dtype=pl.List(pl.Utf8),
            )
            .alias("_matched_author_ids")
        )
    )

    return (
        joined.filter(pl.col("_matched_author_ids").list.len() > 0)
        .explode("_matched_author_ids")
        .select(
            "application_id",
            pl.col("_matched_author_ids").alias("oa_author_id"),
            pl.lit("refs_publication").alias("match_type"),
        )
        .unique()
    )

## 5. Combine matches with priority

The two sources can occasionally produce a match for the same application. Because ORCID codes are unique, ORCID matches are prioritized over reference matching whenever both match on an application.

In [6]:
def _standardize_match_frame(frame: pl.DataFrame) -> pl.DataFrame:
    """
    Keep the match columns needed for combining sources
    """
    return frame.select(
        "application_id",
        "oa_author_id",
        "match_type",
    )


def combine_matches(frames: Sequence[pl.DataFrame]) -> pl.DataFrame:
    """
    Combine match sources and keep the highest confidence match per application.

    Priority: orcid > refs_publication
    """
    # Ignore empty sources so this runs even when no matches are found
    prepared = [_standardize_match_frame(frame) for frame in frames if not frame.is_empty()]
    if not prepared:
        return pl.DataFrame(
            schema={
                "application_id": pl.Int64,
                "oa_author_id": pl.Utf8,
                "match_type": pl.Utf8
        }
    )

    priority_frame = pl.DataFrame(
        {
            "match_type": list(MATCH_PRIORITY.keys()),
            "_priority": list(MATCH_PRIORITY.values()),
        }
    )

    # Sort by application and match confidence and then select highest confidence
    return (
        pl.concat(prepared, how="diagonal_relaxed")
        .join(priority_frame, on="match_type", how="left")
        .with_columns(
            pl.col("_priority").fill_null(999)
        )
        .sort(
            ["application_id", "_priority"],
            descending=[False, False],
        )
        .group_by("application_id", maintain_order=True)
        .first()
        .drop("_priority")
    )

## 6. Run the matching pipeline

The two sources are run separately and then combined. This keeps each source easy to inspect before the priority rule is applied.

In [7]:
# Highest-confidence: shared ORCID
matches_orcid = match_orcid(applications, authors)

# Second-highest: cited works whose author lists identify the applicant
matches_refs = match_reference_works(references, works, applications)

# Apply priority rule
all_matches = combine_matches(
    [
        matches_orcid,
        matches_refs,
    ]
)

all_matches.sort("application_id")

application_id,oa_author_id,match_type
i64,str,str
1,"""A1""","""orcid"""
2,"""A3""","""refs_publication"""
7,"""A6""","""refs_publication"""


## 7. Match coverage summary

This reviews the matching results, shown by type. 

In [8]:
summary = (
    all_matches
    .group_by("match_type")
    .agg(pl.len().alias("n_applications"))
    .sort("match_type")
)

coverage = all_matches["application_id"].n_unique() / applications["application_id"].n_unique()
print(f"Matched applications: {all_matches['application_id'].n_unique()} / {applications['application_id'].n_unique()} ({coverage:.0%})")
summary

Matched applications: 3 / 7 (43%)


match_type,n_applications
str,u32
"""orcid""",1
"""refs_publication""",2


Applications `3`, `4`, `5`, and `6` are intentionally unmatched. They cover four different rejection paths: ambiguous authorship, no matching work title, no available evidence, and a last-name hit with the wrong first initial.

## 8. Validation 

In [9]:
# These are the expected matches
expected = {
    1: ("A1", "orcid"),
    2: ("A3", "refs_publication"),
    7: ("A6", "refs_publication"),
}

# The observed matches
observed = {
    row["application_id"]: (row["oa_author_id"], row["match_type"])
    for row in all_matches.to_dicts()
}

# If the matching worked, then the observed and expected matches should be the same
assert observed == expected

# Ensure that those that should not be matched were not matched
for application_id in [3, 4, 5, 6]:
    assert application_id not in observed

print("Validation passed.")


Validation passed.


## Relation to full project

In the original project, the same functions operate on larger Polars dataframes:

- Applications come from grant application records.
- Authors and works datasets come from OpenAlex parquet files.
- Reference titles are extracted from application text before matching.